In [47]:
import copy
import random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import seaborn as sns
import matplotlib.pyplot as plt

from torch.utils.data import TensorDataset, DataLoader
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# Import Dataset and Tidy

In [48]:
train_data = pd.read_csv("./data/covid.train.csv")
test_data = pd.read_csv("./data/covid.test.csv")

In [49]:
test_data

,id,AL,AK,AZ,AR,CA,CO,CT,FL,GA,...,shop.2,restaurant.2,spent_time.2,large_event.2,public_transit.2,anxious.2,depressed.2,felt_isolated.2,worried_become_ill.2,worried_finances.2
0,0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,52.071090,8.624001,29.374792,5.391413,2.754804,19.695098,13.685645,24.747837,66.194950,44.873473
1,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,58.742461,21.720187,41.375784,9.450179,3.150088,22.075715,17.302077,23.559622,57.015009,38.372829
2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,59.109045,20.123959,40.072556,8.781522,2.888209,23.920870,18.342506,24.993341,55.291498,38.907257
3,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,55.442267,16.083529,36.977612,5.199286,2.575347,21.073800,12.087171,18.608723,67.036197,43.142779
4,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,60.588783,19.503010,42.631236,11.549771,8.530551,15.896575,11.781634,15.065228,61.196518,43.574676
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
888,888,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,56.762931,21.494159,44.202567,14.996865,2.291745,17.740003,12.822676,18.123344,60.417531,37.156229
889,889,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,57.888461,16.770893,37.373472,7.169675,2.631595,20.587449,15.960166,23.710310,58.758735,38.673787
890,890,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,57.589848,16.761311,36.874822,11.046907,1.912310,16.800220,13.280423,22.423640,60.934851,43.122513
891,891,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,57.966384,22.696669,45.350415,20.343487,2.385330,16.528265,15.092539,17.476063,54.862386,44.016255


In [50]:
train_data

,id,AL,AK,AZ,AR,CA,CO,CT,FL,GA,...,restaurant.2,spent_time.2,large_event.2,public_transit.2,anxious.2,depressed.2,felt_isolated.2,worried_become_ill.2,worried_finances.2,tested_positive.2
0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,23.812411,43.430423,16.151527,1.602635,15.409449,12.088688,16.702086,53.991549,43.604229,20.704935
1,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,23.682974,43.196313,16.123386,1.641863,15.230063,11.809047,16.506973,54.185521,42.665766,21.292911
2,2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,23.593983,43.362200,16.159971,1.677523,15.717207,12.355918,16.273294,53.637069,42.972417,21.166656
3,3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,22.576992,42.954574,15.544373,1.578030,15.295650,12.218123,16.045504,52.446223,42.907472,19.896607
4,4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,22.091433,43.290957,15.214655,1.641667,14.778802,12.417256,16.134238,52.560315,43.321985,20.178428
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2695,2695,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,15.090116,30.839219,7.849525,1.760094,14.617563,11.163213,18.742673,68.024690,38.920206,13.008853
2696,2696,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,14.779264,30.617100,7.754800,1.780730,14.513419,11.281241,18.539741,67.855755,39.224244,12.725638
2697,2697,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,14.961085,30.595194,7.744075,1.921828,14.160990,11.163526,18.702564,67.731162,38.740651,12.613441
2698,2698,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,14.609582,30.420998,7.687974,1.992580,14.409427,11.330301,19.134697,67.795100,38.595125,12.477227


## Baseline 0
先用 sklearn 快速確認「資料整理 + 評估流程」都正確。

In [51]:
# 從 training dataset 中分離出 validation dataset
X = train_data.drop(columns=['id', 'tested_positive.2'])
y = train_data['tested_positive.2'] # day3 percentage
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 標準化
scaler = StandardScaler() # 建立一個「標準化器」
X_train_s = scaler.fit_transform(X_train)

# 沿用上方平均數與標準差來校正 val
X_val_s = scaler.transform(X_val)


# ------------------------------------
# baseline 0 (如果我們預測出來 day3 = day2，loss 會是多少？)
pred_b0 = X_val["tested_positive.1"].to_numpy(dtype=np.float64) 
loss_b0 = mean_squared_error(y_val, pred_b0)
print(f"MSE of baseline 0 (predict y3=y2): {loss_b0}")

MSE of baseline 0 (predict y3=y2): 1.0638112270372913


# Baseline 1
建立 Ridge MODEL（y = wx+b），alpha 是懲罰 ｗ 機制，讓 w 不要太大 -> alpha 越小，model 越接近線性模型，擬合更貼但更可能 overfit；alpha 越大，model 越保守較不容易 overfit


In [52]:
# 測試不同 alpha 的 model

best_alpha_loss = {"alpha":0, "loss":10}

for a in list(np.arange(0.01, 2.0001, 0.01)):
    model = Ridge(alpha=a)
    model.fit(X_train_s, y_train)
    pred = model.predict(X_val_s)
    loss = mean_squared_error(y_val, pred)
    if loss < best_alpha_loss["loss"] and loss < loss_b0:
        best_alpha_loss["alpha"] = a
        best_alpha_loss["loss"] = loss

print(f"best alpha: {best_alpha_loss['alpha']}, MSE of baseline 1: {best_alpha_loss['loss']}")

best alpha: 0.01, MSE of baseline 1: 0.9291660509752497


# PyTorch

### helper functions

In [ ]:
# We split the selection budget into two parts:
# a small quota for state columns, and the rest for survey columns.
# This keeps state identity from disappearing without forcing the whole block in.

def set_seed(seed: int, deterministic: bool = True):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        if deterministic:
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False


STATE_ABBR = [
    'AL', 'AK', 'AZ', 'AR', 'CA', 'CO', 'CT', 'FL', 'GA', 'ID',
    'IL', 'IN', 'IA', 'KS', 'KY', 'LA', 'MD', 'MA', 'MI', 'MN',
    'MS', 'MO', 'NE', 'NV', 'NJ', 'NM', 'NY', 'NC', 'OH', 'OK',
    'OR', 'PA', 'RI', 'SC', 'TX', 'UT', 'VA', 'WA', 'WV', 'WI',
]
SURVEY_COLS = [
    'cli', 'ili', 'hh_cmnty_cli', 'nohh_cmnty_cli', 'wearing_mask',
    'travel_outside_state', 'work_outside_home', 'shop', 'restaurant',
    'spent_time', 'large_event', 'public_transit', 'anxious', 'depressed',
    'felt_isolated', 'worried_become_ill', 'worried_finances',
]
STATE_COLS = [f'state_{abbr}' for abbr in STATE_ABBR]
DAY1_TP_COL = 'd1_tested_positive'
DAY2_TP_COL = 'd2_tested_positive'
TARGET_COL = 'd3_tested_positive'
# SELECT_K controls only the selectable budget for state + survey columns.
# day1/day2 tested_positive stay outside this budget because they are known baselines.
SELECT_K = 16
STATE_RATIO = 0.20
STATE_MIN_KEEP = 2
STATE_MAX_KEEP = 4
TRAIN_SEEDS = [2025, 2026, 2027]
FINAL_SEEDS = [2025, 2026, 2027]
BLEND_GRID = np.linspace(0.0, 1.0, 21)


def rename_covid_columns(df, has_day3_target: bool):
    renamed = ['id'] + STATE_COLS
    for day in (1, 2, 3):
        renamed.extend([f'd{day}_{name}' for name in SURVEY_COLS])
        if day < 3 or has_day3_target:
            renamed.append(f'd{day}_tested_positive')
    out = df.copy()
    out.columns = renamed
    return out


def state_ids(df):
    return df[STATE_COLS].to_numpy(dtype=np.float32).argmax(axis=1)


def split_state_timewise(df, val_frac=0.2, min_val_per_state=10):
    sids = state_ids(df)
    tr_idx = []
    va_idx = []
    for s in range(len(STATE_COLS)):
        idx = np.flatnonzero(sids == s)
        if len(idx) == 0:
            continue
        val_count = max(min_val_per_state, int(round(len(idx) * val_frac)))
        val_count = min(val_count, len(idx) - 1)
        tr_idx.extend(idx[:-val_count].tolist())
        va_idx.extend(idx[-val_count:].tolist())
    return df.iloc[tr_idx].copy(), df.iloc[va_idx].copy()


def scale_column(train_df, val_df, col):
    scaler = StandardScaler()
    train_s = scaler.fit_transform(train_df[[col]]).astype(np.float32)
    val_s = scaler.transform(val_df[[col]]).astype(np.float32)
    return scaler, train_s, val_s


def select_top_k_columns(train_df, y_train, candidate_cols, k):
    k = min(int(k), len(candidate_cols))
    selector = SelectKBest(score_func=f_regression, k=k)
    selector.fit(train_df[candidate_cols], y_train)
    return [col for col, keep in zip(candidate_cols, selector.get_support()) if keep]


def scale_selected_columns(train_df, val_df, selected_cols):
    scaler = StandardScaler()
    train_s = scaler.fit_transform(train_df[selected_cols]).astype(np.float32)
    val_s = scaler.transform(val_df[selected_cols]).astype(np.float32)
    return scaler, train_s, val_s

def state_keep_for_budget(total_k: int):
    if total_k <= 1:
        return 1
    keep = int(round(total_k * STATE_RATIO))
    keep = max(STATE_MIN_KEEP, keep)
    keep = min(STATE_MAX_KEEP, keep)
    keep = min(keep, len(STATE_COLS), total_k - 1)
    return max(1, keep)


def make_tensor_loader(X_s, y, batch_size: int, shuffle: bool, seed: int):
    dataset = TensorDataset(
        torch.tensor(X_s, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32).reshape(-1, 1),
    )
    generator = torch.Generator()
    generator.manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator,
        num_workers=0,
    )


class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden=64, dropout=0.15, depth=2):
        super().__init__()
        layers = []
        dim = input_dim
        for _ in range(max(1, depth)):
            layers.extend([
                nn.Linear(dim, hidden),
                nn.LayerNorm(hidden),
                nn.GELU(),
                nn.Dropout(dropout),
            ])
            dim = hidden
        layers.append(nn.Linear(dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def evaluate_mse(model, loader, device):
    model.eval()
    preds = []
    targets = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            preds.append(model(xb))
            targets.append(yb)
    pred = torch.cat(preds, dim=0)
    target = torch.cat(targets, dim=0)
    return F.mse_loss(pred, target).item()


@torch.no_grad()
def predict_numpy(model, X_s, device, batch_size=256):
    model.eval()
    loader = DataLoader(
        TensorDataset(torch.tensor(X_s, dtype=torch.float32)),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )
    preds = []
    for (xb,) in loader:
        preds.append(model(xb.to(device)).cpu().numpy())
    return np.vstack(preds).reshape(-1)


def train_simple_mlp(
    model,
    train_loader,
    train_eval_loader,
    val_loader,
    device,
    epochs=300,
    lr=1e-3,
    weight_decay=1e-3,
    patience=40,
    log_every=25,
):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=10,
        min_lr=1e-5,
    )

    best_state = None
    best_epoch = 0
    best_val = float('inf')
    bad_count = 0
    train_hist = []
    val_hist = []

    model.to(device)

    for epoch in range(1, epochs + 1):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            pred = model(xb)
            loss = F.mse_loss(pred, yb)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        train_mse = evaluate_mse(model, train_eval_loader, device)
        val_mse = evaluate_mse(model, val_loader, device)
        train_hist.append(train_mse)
        val_hist.append(val_mse)
        scheduler.step(val_mse)

        if val_mse < best_val - 1e-8:
            best_val = val_mse
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            bad_count = 0
        else:
            bad_count += 1
            if bad_count >= patience:
                print(f'Early stopping at epoch {epoch}. Best epoch = {best_epoch}, best va_mse = {best_val:.4f}')
                break

        if epoch <= 5 or epoch % log_every == 0 or val_mse <= best_val + 1e-8:
            print(f'epoch {epoch}: tr_mse={train_mse:.4f}, va_mse={val_mse:.4f}')

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, train_hist, val_hist, best_epoch, best_val


def fit_full_mlp(
    model,
    train_loader,
    device,
    epochs=200,
    lr=1e-3,
    weight_decay=1e-3,
    log_every=25,
):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    model.to(device)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_n = 0
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            pred = model(xb)
            loss = F.mse_loss(pred, yb)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += float(loss.item()) * len(xb)
            total_n += len(xb)

        train_mse = total_loss / max(total_n, 1)
        if epoch <= 5 or epoch % log_every == 0 or epoch == epochs:
            print(f'final epoch {epoch}/{epochs}: train_mse={train_mse:.4f}')

    return model


def train_one_run(
    X_train_s,
    y_train,
    X_val_s,
    y_val,
    seed,
    device,
    hidden=64,
    depth=2,
    dropout=0.15,
    lr=1e-3,
    weight_decay=1e-3,
    epochs=300,
    patience=40,
    batch_size=64,
):
    set_seed(seed)
    train_loader = make_tensor_loader(X_train_s, y_train, batch_size=batch_size, shuffle=True, seed=seed)
    train_eval_loader = make_tensor_loader(X_train_s, y_train, batch_size=256, shuffle=False, seed=seed)
    val_loader = make_tensor_loader(X_val_s, y_val, batch_size=256, shuffle=False, seed=seed)

    model = SimpleMLP(input_dim=X_train_s.shape[1], hidden=hidden, dropout=dropout, depth=depth)
    model, train_hist, val_hist, best_epoch, best_val = train_simple_mlp(
        model,
        train_loader=train_loader,
        train_eval_loader=train_eval_loader,
        val_loader=val_loader,
        device=device,
        epochs=epochs,
        lr=lr,
        weight_decay=weight_decay,
        patience=patience,
    )

    val_pred = predict_numpy(model, X_val_s, device)
    return {
        'seed': seed,
        'model': model,
        'train_hist': train_hist,
        'val_hist': val_hist,
        'best_epoch': best_epoch,
        'best_val': best_val,
        'val_pred': val_pred,
    }


def fit_seed_ensemble(
    X_train_s,
    y_train,
    X_val_s,
    y_val,
    seed_list,
    device,
    hidden=64,
    depth=2,
    dropout=0.15,
    lr=1e-3,
    weight_decay=1e-3,
    epochs=300,
    patience=40,
    batch_size=64,
):
    runs = []
    val_preds = []
    for seed in seed_list:
        run = train_one_run(
            X_train_s=X_train_s,
            y_train=y_train,
            X_val_s=X_val_s,
            y_val=y_val,
            seed=seed,
            device=device,
            hidden=hidden,
            depth=depth,
            dropout=dropout,
            lr=lr,
            weight_decay=weight_decay,
            epochs=epochs,
            patience=patience,
            batch_size=batch_size,
        )
        runs.append(run)
        val_preds.append(run['val_pred'])
        print(f"seed {seed}: best_epoch={run['best_epoch']}, best_val={run['best_val']:.4f}")

    ensemble_val_pred = np.mean(np.stack(val_preds, axis=0), axis=0)
    ensemble_val_mse = mean_squared_error(y_val, ensemble_val_pred)
    avg_best_epoch = float(np.mean([run['best_epoch'] for run in runs])) if runs else 0.0
    return runs, ensemble_val_pred, ensemble_val_mse, avg_best_epoch


def train_full_model(
    X_train_s,
    y_train,
    X_test_s,
    seed,
    device,
    hidden=64,
    depth=2,
    dropout=0.15,
    lr=1e-3,
    weight_decay=1e-3,
    epochs=200,
    batch_size=64,
):
    set_seed(seed)
    train_loader = make_tensor_loader(X_train_s, y_train, batch_size=batch_size, shuffle=True, seed=seed)
    model = SimpleMLP(input_dim=X_train_s.shape[1], hidden=hidden, dropout=dropout, depth=depth)
    model = fit_full_mlp(
        model,
        train_loader=train_loader,
        device=device,
        epochs=epochs,
        lr=lr,
        weight_decay=weight_decay,
    )
    test_pred = predict_numpy(model, X_test_s, device)
    return model, test_pred


def choose_blend_weight(y_true, pred_a, pred_b, grid=None):
    if grid is None:
        grid = BLEND_GRID
    y_true = np.asarray(y_true, dtype=np.float32).reshape(-1)
    pred_a = np.asarray(pred_a, dtype=np.float32).reshape(-1)
    pred_b = np.asarray(pred_b, dtype=np.float32).reshape(-1)
    best_w = 1.0
    best_score = float('inf')
    for w in grid:
        pred = w * pred_a + (1.0 - w) * pred_b
        score = mean_squared_error(y_true, pred)
        if score < best_score:
            best_score = score
            best_w = float(w)
    return best_w, best_score

# Quota-Based Feature Selection

這版還是走你驗證過的 flat 路線，但把選擇規則改成配額制：
- state 永遠保留一小部分名額，不會被 survey 完全擠掉
- survey 仍然拿走大部分名額，避免 state 佔太多空間
- day1 / day2 tested_positive 仍然固定保留，因為它們是最強的已知 baseline
- 再用幾個 seed 的同一個小 MLP 做平均

In [ ]:
train_df = rename_covid_columns(train_data, has_day3_target=True)
test_df = rename_covid_columns(test_data, has_day3_target=False)

train_part_df, val_part_df = split_state_timewise(train_df, val_frac=0.2, min_val_per_state=10)
y_train_part = train_part_df[TARGET_COL].to_numpy(dtype=np.float32)
y_val_part = val_part_df[TARGET_COL].to_numpy(dtype=np.float32)
day2_val_part = val_part_df[DAY2_TP_COL].to_numpy(dtype=np.float32)

survey_cols = [f'd{day}_{name}' for day in (1, 2, 3) for name in SURVEY_COLS]
selection_candidate_cols = STATE_COLS + survey_cols
state_keep = state_keep_for_budget(SELECT_K)
survey_keep = SELECT_K - state_keep

print(f'train_df shape = {train_df.shape}')
print(f'test_df shape = {test_df.shape}')
print(f'train_part rows = {len(train_part_df)}, val_part rows = {len(val_part_df)}')
print(f'selection candidate columns = {len(selection_candidate_cols)}')
print(f'state quota = {state_keep}')
print(f'survey quota = {survey_keep}')
print(f'survey columns = {len(survey_cols)}')
print(f'day2 baseline val MSE = {mean_squared_error(y_val_part, day2_val_part):.4f}')

## Feature Selection

State columns get their own small quota.
Survey columns get the rest of the budget.
This keeps state identity present without forcing the full state block into the model.

In [ ]:
selected_state_cols = select_top_k_columns(
    train_part_df,
    y_train_part,
    STATE_COLS,
    state_keep,
)
selected_survey_cols = select_top_k_columns(
    train_part_df,
    y_train_part,
    survey_cols,
    survey_keep,
)

# day1/day2 are added after selection because they are fixed baseline signals.
final_feature_cols = selected_state_cols + selected_survey_cols + [DAY1_TP_COL, DAY2_TP_COL]
X_train_s, X_val_s = scale_selected_columns(train_part_df, val_part_df, final_feature_cols)

print(f'SELECT_K = {SELECT_K}')
print(f'state quota = {state_keep}')
print(f'survey quota = {survey_keep}')
print(f'selected state feature count = {len(selected_state_cols)}')
print(f'selected survey feature count = {len(selected_survey_cols)}')
print(f'final feature count = {len(final_feature_cols)}')
print(f'X_train_s shape = {X_train_s.shape}')
print(f'X_val_s shape = {X_val_s.shape}')
print('selected state columns:')
print(selected_state_cols)
print('selected survey columns:')
print(selected_survey_cols)
print('final feature columns:')
print(final_feature_cols)

# Diagnostic only: compare against the older idea where every candidate column competes for the same k slots.
# This makes it easy to see whether state one-hot columns get pushed out by the survey features.
all_candidate_cols = STATE_COLS + survey_cols + [DAY1_TP_COL, DAY2_TP_COL]
all_selector = SelectKBest(score_func=f_regression, k=min(SELECT_K, len(all_candidate_cols)))
all_selector.fit(train_part_df[all_candidate_cols], y_train_part)
all_scores = pd.DataFrame({
    'feature': all_candidate_cols,
    'score': all_selector.scores_,
})
all_scores['score'] = all_scores['score'].replace([np.inf, -np.inf], np.nan).fillna(-np.inf)
all_scores = all_scores.sort_values(['score', 'feature'], ascending=[False, True]).reset_index(drop=True)
all_selected = all_scores.head(SELECT_K)['feature'].tolist()
state_selected = [col for col in all_selected if col in STATE_COLS]
state_score_table = all_scores[all_scores['feature'].isin(STATE_COLS)]
best_state_score = float(state_score_table['score'].max()) if len(state_score_table) > 0 else float('nan')

print('\nDiagnostic: if SelectKBest sees every candidate feature at once')
print(f'all-column top-{SELECT_K} selected count = {len(all_selected)}')
print(f'state columns in top-{SELECT_K} = {len(state_selected)}')
print(f'best state score = {best_state_score:.4f}')
print('Top 25 all-column features by f-score:')
print(all_scores.head(25).to_string(index=False))
print('State columns ranked by f-score:')
print(state_score_table.to_string(index=False))

## Train MLP

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
seed = 2025
set_seed(seed)

hidden = 64
depth = 2
dropout = 0.15
lr = 1e-3
weight_decay = 1e-3
epochs = 300
patience = 40
batch_size = 64

runs, ensemble_val_pred, ensemble_val_mse, avg_best_epoch = fit_seed_ensemble(
    X_train_s=X_train_s,
    y_train=y_train_part,
    X_val_s=X_val_s,
    y_val=y_val_part,
    seed_list=TRAIN_SEEDS,
    device=device,
    hidden=hidden,
    depth=depth,
    dropout=dropout,
    lr=lr,
    weight_decay=weight_decay,
    epochs=epochs,
    patience=patience,
    batch_size=batch_size,
)

blend_weight, blend_val_mse = choose_blend_weight(y_val_part, ensemble_val_pred, day2_val_part)
best_run = min(runs, key=lambda r: r['best_val'])
train_hist = best_run['train_hist']
val_hist = best_run['val_hist']
best_epoch = best_run['best_epoch']
best_val = best_run['best_val']

print(f'best single run best_epoch = {best_epoch}, best_val = {best_val:.4f}')
print(f'ensemble val MSE = {ensemble_val_mse:.4f}')
print(f'blend weight vs. day2 = {blend_weight:.2f}')
print(f'blended val MSE = {blend_val_mse:.4f}')
print(f'outer avg best epoch = {avg_best_epoch:.1f}')

## Epoch Curve

這張圖只顯示其中一個 seed 的訓練過程，方便看模型有沒有正常收斂。
真正用來送分的是幾個 seed 的平均，再和 day2 baseline 混合。

In [ ]:
sns.set_theme(style='whitegrid')
epochs_ran = np.arange(1, len(train_hist) + 1)
best_epoch_plot = best_epoch if best_epoch is not None else int(np.argmin(val_hist) + 1)
best_va_mse = val_hist[best_epoch_plot - 1]

plt.figure(figsize=(9, 5))
plt.plot(epochs_ran, train_hist, label='Train MSE', linewidth=2)
plt.plot(epochs_ran, val_hist, label='Validation MSE', linewidth=2)
plt.axvline(best_epoch_plot, color='tab:red', linestyle='--', alpha=0.35, label=f'Best epoch = {best_epoch_plot}')
plt.scatter(best_epoch_plot, best_va_mse, color='tab:red', s=35, zorder=3)
plt.annotate(
    f'best va_mse = {best_va_mse:.4f}',
    xy=(best_epoch_plot, best_va_mse),
    xytext=(12, 14),
    textcoords='offset points',
    fontsize=10,
    color='tab:red',
    bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='tab:red', alpha=0.9),
    arrowprops=dict(arrowstyle='->', color='tab:red', lw=1),
)
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.title(f'Training / Validation Day3 MSE vs. Epoch | best va_mse = {best_va_mse:.4f}')
plt.legend()
plt.tight_layout()
plt.show()

## Final Fit and Submission

這裡只做一次和訓練階段完全相同的事情：
- state 和 survey 都沿用剛剛的配額邏輯
- 在完整 training data 上重訓幾個 seed
- 平均 test prediction
- 再和 day2 baseline 混合

In [ ]:
full_train_df = rename_covid_columns(train_data, has_day3_target=True)
full_test_df = rename_covid_columns(test_data, has_day3_target=False)
y_full = full_train_df[TARGET_COL].to_numpy(dtype=np.float32)
day2_test_pred = full_test_df[DAY2_TP_COL].to_numpy(dtype=np.float32)

final_feature_cols = selected_state_cols + selected_survey_cols + [DAY1_TP_COL, DAY2_TP_COL]
X_full_train_s, X_full_test_s = scale_selected_columns(full_train_df, full_test_df, final_feature_cols)

print(f'Final selected state columns = {selected_state_cols}')
print(f'Final selected survey columns = {selected_survey_cols}')
print(f'Final feature columns = {final_feature_cols}')
print(f'Final input dim = {X_full_train_s.shape[1]}')

final_epochs = max(100, int(round(avg_best_epoch * 1.15)))
print(f'Final training epochs per seed = {final_epochs}')
print(f'Final seed list = {FINAL_SEEDS}')

final_test_preds = []
for seed in FINAL_SEEDS:
    print()
    print(f'Training final model with seed {seed}')
    _, test_pred = train_full_model(
        X_train_s=X_full_train_s,
        y_train=y_full,
        X_test_s=X_full_test_s,
        seed=seed,
        device=device,
        hidden=hidden,
        depth=depth,
        dropout=dropout,
        lr=lr,
        weight_decay=weight_decay,
        epochs=final_epochs,
        batch_size=batch_size,
    )
    final_test_preds.append(test_pred)

ensemble_test_pred = np.mean(np.stack(final_test_preds, axis=0), axis=0)
final_test = blend_weight * ensemble_test_pred + (1.0 - blend_weight) * day2_test_pred
final_test = np.clip(final_test, 0.0, 100.0)

submission = pd.DataFrame({
    'id': full_test_df['id'].to_numpy(),
    'tested_positive': final_test,
})
submission.to_csv('submission.csv', index=False)

import subprocess
subprocess.run([
    'kaggle', 'competitions', 'submit',
    '-c', 'ml2021spring-hw1',
    '-f', 'submission.csv',
    '-m', f'quota-based SelectKBest k={SELECT_K} seeds={len(FINAL_SEEDS)}',
], check=True)

print(f'Final blend weight = {blend_weight:.2f}')
print('Saved submission.csv and submitted to Kaggle')
submission.head()